# Notebook 01 - Data Collection & Merging

This notebook loads all raw data sources and merges them into a single clean dataset
for 35 countries, saved to `data/processed/merged_data.csv`.

**Sources used:**
| File | What it provides |
|---|---|
| `cyber_security_kaggle.csv` | Overall GCI score, NCSI score, DDL score |
| `ITU_GCI.csv` | GCI pillar scores: Technical, Legal, Organisational, Capacity, Cooperation |
| `worldbank_internet.csv` | Internet penetration (% of population) |
| `worldbank_gdp.csv` | GDP per capita (current USD) - used in regression |
| `HDR_HDI.xlsx` | Human Development Index - used in regression |

In [15]:
import pandas as pd
import numpy as np
import os
from IPython.display import display

RAW       = '../data/raw/'
PROCESSED = '../data/processed/'
os.makedirs(PROCESSED, exist_ok=True)

# 35 target countries
TARGET_COUNTRIES = [
    'United States', 'United Kingdom', 'Germany', 'France', 'Estonia',
    'Finland', 'Norway', 'Sweden', 'Denmark', 'Netherlands',
    'Australia', 'Canada', 'Japan', 'South Korea', 'Singapore',
    'Israel', 'Switzerland', 'Austria', 'Belgium', 'Spain',
    'Italy', 'Portugal', 'Poland', 'Czech Republic', 'Hungary',
    'Brazil', 'Mexico', 'Argentina', 'Turkey', 'China',
    'India', 'South Africa', 'Nigeria', 'Kenya', 'Ghana',
]

# ISO3 codes → country name (for joining ITU and World Bank data)
ISO3_TO_COUNTRY = {
    'USA': 'United States',   'GBR': 'United Kingdom', 'DEU': 'Germany',
    'FRA': 'France',          'EST': 'Estonia',         'FIN': 'Finland',
    'NOR': 'Norway',          'SWE': 'Sweden',          'DNK': 'Denmark',
    'NLD': 'Netherlands',     'AUS': 'Australia',       'CAN': 'Canada',
    'JPN': 'Japan',           'KOR': 'South Korea',     'SGP': 'Singapore',
    'ISR': 'Israel',          'CHE': 'Switzerland',     'AUT': 'Austria',
    'BEL': 'Belgium',         'ESP': 'Spain',           'ITA': 'Italy',
    'PRT': 'Portugal',        'POL': 'Poland',          'CZE': 'Czech Republic',
    'HUN': 'Hungary',         'BRA': 'Brazil',          'MEX': 'Mexico',
    'ARG': 'Argentina',       'TUR': 'Turkey',          'CHN': 'China',
    'IND': 'India',           'ZAF': 'South Africa',    'NGA': 'Nigeria',
    'KEN': 'Kenya',           'GHA': 'Ghana',
}

# World Bank uses slightly different names for some countries
WB_NAME_FIX = {
    'Czechia':                'Czech Republic',
    'Korea, Rep.':            'South Korea',
    'Turkiye':                'Turkey',
    'Egypt, Arab Rep.':       'Egypt',
    'Iran, Islamic Rep.':     'Iran',
    "Cote d'Ivoire":          "Cote d'ivoire",
}

print('Setup complete.')

Setup complete.


## 1. Kaggle Dataset - Overall GCI, NCSI, DDL scores

In [16]:
kaggle_raw = pd.read_csv(f'{RAW}cyber_security_kaggle.csv')
print(f'Kaggle raw shape: {kaggle_raw.shape}')
display(kaggle_raw.head())

Kaggle raw shape: (192, 6)


,Country,Region,CEI,GCI,NCSI,DDL
0,Afghanistan,Asia-Pasific,1.000,5.20,11.69,19.50
1,Albania,Europe,0.566,64.32,62.34,48.74
2,Algeria,Africa,0.721,33.95,33.77,42.81
3,Andorra,Europe,NaN,26.38,NaN,NaN
4,Angola,Africa,NaN,12.99,9.09,22.69


In [17]:
kaggle = kaggle_raw[['Country', 'GCI', 'NCSI', 'DDL']].rename(columns={
    'Country': 'country',
    'GCI':     'gci_overall',
    'NCSI':    'ncsi_score',
    'DDL':     'ddl_score',
})
kaggle['country'] = kaggle['country'].str.strip()

# Filter to our 35 countries
kaggle = kaggle[kaggle['country'].isin(TARGET_COUNTRIES)].reset_index(drop=True)
print(f'Countries matched from Kaggle: {len(kaggle)}/35')
missing = set(TARGET_COUNTRIES) - set(kaggle['country'])
if missing:
    print(f'Missing: {missing}')
display(kaggle)

Countries matched from Kaggle: 35/35


,country,gci_overall,ncsi_score,ddl_score
0,Argentina,50.12,63.64,60.43
1,Australia,97.47,66.23,77.61
2,Austria,93.89,68.83,75.76
3,Belgium,96.25,94.81,74.07
4,Brazil,96.60,51.95,59.11
5,Canada,97.67,70.13,75.96
6,China,92.53,51.95,62.41
7,Czech Republic,74.37,92.21,69.21
8,Denmark,92.60,84.42,82.68
9,Estonia,99.48,93.51,75.59


## 2. ITU GCI - Pillar Scores (Technical, Legal, Organisational, Capacity, Cooperation)

The ITU file is in long format (one row per country-indicator). We pivot it to get one column per pillar.

In [18]:
itu_raw = pd.read_csv(f'{RAW}ITU_GCI.csv')

# The 5 pillar score indicators we need
PILLAR_INDICATORS = {
    'ITU_GCI_TECH_SCORE': 'gci_technical',
    'ITU_GCI_LEGL_SCRE':  'gci_legal',
    'ITU_GCI_ORG_SCORE':  'gci_organisational',
    'ITU_GCI_CDS_SCORE':  'gci_capacity',
    'ITU_GCI_COOP_SCORE': 'gci_cooperation',
}

# Filter to pillar rows only
itu_pillars = itu_raw[itu_raw['INDICATOR'].isin(PILLAR_INDICATORS.keys())].copy()

# Map ISO3 to our country names
itu_pillars['country'] = itu_pillars['REF_AREA'].map(ISO3_TO_COUNTRY)
itu_pillars = itu_pillars.dropna(subset=['country'])

# Rename indicator codes to readable column names
itu_pillars['indicator_name'] = itu_pillars['INDICATOR'].map(PILLAR_INDICATORS)

# Pivot to wide format
itu_wide = itu_pillars.pivot_table(
    index='country', columns='indicator_name', values='OBS_VALUE'
).reset_index()
itu_wide.columns.name = None

# Pillar SCORE indicators are on the ITU raw aggregated scale (not 0-1).
# Do not rescale here - min-max normalisation happens in notebook 05.

print(f'ITU pillar data shape: {itu_wide.shape}')
display(itu_wide.head())

ITU pillar data shape: (35, 6)


,country,gci_capacity,gci_cooperation,gci_legal,gci_organisational,gci_technical
0,Argentina,3.135239,9.983615,14.553914,10.186348,12.962927
1,Australia,19.720586,19.127009,20.000000,19.490706,18.516867
2,Austria,18.735104,17.143173,18.924699,17.599348,19.115610
3,Belgium,19.596973,20.000000,20.000000,17.884250,19.055728
4,Brazil,19.367390,19.703050,20.000000,18.137006,19.363917


## 3. World Bank - Internet Penetration

In [19]:
def load_worldbank(filepath, value_col_name, year_range=range(2018, 2024)):
    """Load a World Bank CSV and extract the most recent non-null year per country."""
    df = pd.read_csv(filepath, skiprows=4)
    year_cols = [str(y) for y in year_range if str(y) in df.columns]
    df = df[['Country Name', 'Country Code'] + year_cols].rename(
        columns={'Country Name': 'country', 'Country Code': 'iso3'}
    )
    df['country'] = df['country'].str.strip().replace(WB_NAME_FIX)
    # Most recent non-null value
    df[value_col_name] = df[year_cols].apply(
        lambda row: row.dropna().iloc[-1] if not row.dropna().empty else np.nan, axis=1
    )
    return df[['country', value_col_name]]


wb_internet = load_worldbank(f'{RAW}worldbank_internet.csv', 'internet_penetration')
wb_internet = wb_internet[wb_internet['country'].isin(TARGET_COUNTRIES)]
print(f'Internet penetration: {len(wb_internet)} countries')
display(wb_internet.head())

Internet penetration: 35 countries


,country,internet_penetration
9,Argentina,89.228972
13,Australia,96.116096
14,Austria,95.334671
17,Belgium,94.626251
29,Brazil,84.150602


## 4. World Bank - GDP per Capita (regression benchmark only)

In [20]:
wb_gdp = load_worldbank(f'{RAW}worldbank_gdp.csv', 'gdp_per_capita')
wb_gdp = wb_gdp[wb_gdp['country'].isin(TARGET_COUNTRIES)]
print(f'GDP per capita: {len(wb_gdp)} countries')
display(wb_gdp.head())

GDP per capita: 35 countries


,country,gdp_per_capita
9,Argentina,14261.846567
13,Australia,65058.377315
14,Austria,56579.504175
17,Belgium,55291.475454
29,Brazil,10377.589772


## 5. UN HDI (regression benchmark only)

In [21]:
hdi_raw = pd.read_excel(f'{RAW}HDR_HDI.xlsx', header=None)

# Data starts at row 8; column 1 = country name, column 2 = HDI value
hdi_data = hdi_raw.iloc[7:, [1, 2]].copy()
hdi_data.columns = ['country', 'hdi_score']
hdi_data = hdi_data.dropna(subset=['country', 'hdi_score'])

# Remove group header rows (they have non-numeric hdi_score)
hdi_data = hdi_data[pd.to_numeric(hdi_data['hdi_score'], errors='coerce').notna()]
hdi_data['hdi_score'] = pd.to_numeric(hdi_data['hdi_score'])
hdi_data['country'] = hdi_data['country'].astype(str).str.strip()

# Manual name fixes to match our country list
HDI_NAME_FIX = {
    'Korea (Republic of)':      'South Korea',
    'Czechia':                  'Czech Republic',
    'Türkiye':                  'Turkey',
    'United States':            'United States',
    'Iran (Islamic Republic of)': 'Iran',
}
hdi_data['country'] = hdi_data['country'].replace(HDI_NAME_FIX)
hdi_data = hdi_data[hdi_data['country'].isin(TARGET_COUNTRIES)].reset_index(drop=True)

print(f'HDI: {len(hdi_data)} countries matched')
display(hdi_data.head())

HDI: 35 countries matched


,country,hdi_score
0,Switzerland,0.967
1,Norway,0.966
2,Denmark,0.952
3,Sweden,0.952
4,Germany,0.950


## 6. Merge All Sources

The Kaggle dataset is the anchor (it defines which countries we have). Everything else is left-joined onto it.

In [22]:
merged = kaggle.copy()

for df, label in [
    (itu_wide,    'ITU GCI pillars'),
    (wb_internet, 'Internet penetration'),
    (wb_gdp,      'GDP per capita'),
    (hdi_data,    'HDI'),
]:
    merged = merged.merge(df, on='country', how='left')
    print(f'After merging {label}: {merged.shape}')

print(f'\nFinal shape: {merged.shape}')
display(merged)

After merging ITU GCI pillars: (35, 9)
After merging Internet penetration: (35, 10)
After merging GDP per capita: (35, 11)
After merging HDI: (35, 12)

Final shape: (35, 12)


,country,gci_overall,ncsi_score,ddl_score,gci_capacity,gci_cooperation,gci_legal,gci_organisational,gci_technical,internet_penetration,gdp_per_capita,hdi_score
0,Argentina,50.12,63.64,60.43,3.135239,9.983615,14.553914,10.186348,12.962927,89.228972,14261.846567,0.849
1,Australia,97.47,66.23,77.61,19.720586,19.127009,20.000000,19.490706,18.516867,96.116096,65058.377315,0.946
2,Austria,93.89,68.83,75.76,18.735104,17.143173,18.924699,17.599348,19.115610,95.334671,56579.504175,0.926
3,Belgium,96.25,94.81,74.07,19.596973,20.000000,20.000000,17.884250,19.055728,94.626251,55291.475454,0.942
4,Brazil,96.60,51.95,59.11,19.367390,19.703050,20.000000,18.137006,19.363917,84.150602,10377.589772,0.760
5,Canada,97.67,70.13,75.96,19.489315,19.703050,19.451367,20.000000,16.782165,94.145302,54220.328504,0.935
6,China,92.53,51.95,62.41,18.795183,18.304709,20.000000,17.487949,17.542544,90.600000,12951.178240,0.788
7,Czech Republic,74.37,92.21,69.21,13.749763,13.709668,19.444751,17.098907,17.150437,85.994048,31761.594410,0.895
8,Denmark,92.60,84.42,82.68,19.741319,17.944444,19.649831,19.490706,19.471893,98.775592,68043.546697,0.952
9,Estonia,99.48,93.51,75.59,19.611577,20.000000,20.000000,20.000000,17.648276,93.183422,30264.006489,0.899


## 7. Data Quality Summary

In [23]:
print('=== Missing Values per Column ===')
missing = merged.isnull().sum()
pct     = (missing / len(merged) * 100).round(1)
quality = pd.DataFrame({'missing': missing, 'pct_%': pct})
display(quality[quality['missing'] > 0] if quality['missing'].sum() > 0 else 'No missing values!')

print('\n=== Descriptive Statistics ===')
display(merged.describe().round(2))

=== Missing Values per Column ===


'No missing values!'


=== Descriptive Statistics ===


,gci_overall,ncsi_score,ddl_score,gci_capacity,gci_cooperation,gci_legal,gci_organisational,gci_technical,internet_penetration,gdp_per_capita,hdi_score
count,35.00,35.00,35.00,35.00,35.00,35.00,35.00,35.00,35.00,35.00,35.00
mean,92.23,70.39,68.33,18.16,18.70,19.47,18.24,18.76,86.61,39504.48,0.87
std,9.78,17.60,14.37,3.23,2.16,1.15,2.03,1.36,14.99,27000.17,0.11
min,50.12,31.17,31.76,3.14,9.98,14.55,10.19,12.96,32.07,1942.59,0.55
25%,91.10,61.69,61.42,18.18,18.12,19.56,17.54,18.31,85.88,14061.40,0.85
50%,96.25,68.83,75.50,19.37,19.70,20.00,18.89,19.22,91.45,35674.10,0.91
75%,97.55,85.06,79.31,19.76,20.00,20.00,19.49,19.65,95.39,55120.88,0.94
max,100.00,94.81,82.93,20.00,20.00,20.00,20.00,20.00,99.00,100623.55,0.97


## 8. Save Master Dataset

In [24]:
output_path = f'{PROCESSED}merged_data.csv'
merged.to_csv(output_path, index=False)

print(f'Saved to: {output_path}')
print(f'Shape:    {merged.shape}')
print(f'\nColumns:')
for col in merged.columns:
    n_missing = merged[col].isnull().sum()
    print(f'  {col:<30} {n_missing} missing')

Saved to: ../data/processed/merged_data.csv
Shape:    (35, 12)

Columns:
  country                        0 missing
  gci_overall                    0 missing
  ncsi_score                     0 missing
  ddl_score                      0 missing
  gci_capacity                   0 missing
  gci_cooperation                0 missing
  gci_legal                      0 missing
  gci_organisational             0 missing
  gci_technical                  0 missing
  internet_penetration           0 missing
  gdp_per_capita                 0 missing
  hdi_score                      0 missing
